In [ ]:
### Carregar as Libraries
import pandas as pd
import numpy as np
from numpy import mean, std
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict, KFold, train_test_split
from sklearn.metrics import f1_score

# ============================================================
# 1. LISTA DE TODAS AS SUAS BASES
# ============================================================
bases = [
    'HOG_128_8x8.csv',
    'HOG_128_16x16.csv',
    'HOG_128_32x32.csv',
    'HOG_256_8x8.csv',
    'HOG_256_16x16.csv',
    'HOG_256_32x32.csv',
    'LBP_256_3R.csv',
    'LBP_256_6R.csv',
    'LBP_256_12R.csv',
    'PCA_075_HOG_128_8x8.csv',
    'PCA_075_HOG_128_16x16.csv',
    'PCA_075_HOG_256_8x8.csv',
    'PCA_075_HOG_256_16x16.csv',
    'PCA_075_HOG_256_32x32.csv',
    'PCA_090_HOG_128_8x8.csv',
    'PCA_090_HOG_128_16x16.csv',
    'PCA_090_HOG_256_16x16.csv',
    'PCA_090_HOG_256_8x8.csv',
    'PCA_090_HOG_256_32x32.csv',
]

# ============================================================
# 2. CONFIGURAÇÕES
# ============================================================
k_values = range(1, 11)
kf = KFold(n_splits=10, random_state=1, shuffle=True)

resultados = []

# ============================================================
# 3. LOOP PRINCIPAL
# ============================================================
for base_nome in bases:
    print(f"\n{'='*60}")
    print(f"Processando: {base_nome}")
    print(f"{'='*60}")

    dataset = pd.read_csv(base_nome, encoding='utf-8')

    X = dataset.iloc[:, :-1]
    y = dataset.iloc[:, -1]

    # =============================================
    # TRATAMENTO DE VALORES FALTANTES (NaN)
    # =============================================
    # Verificar se há NaN
    nan_count = X.isnull().sum().sum()
    if nan_count > 0:
        print(f"  ⚠️  Encontrados {nan_count} valores NaN! Preenchendo com 0...")
        X = X.fillna(0)

    # Verificar se há NaN no y também
    nan_y = y.isnull().sum()
    if nan_y > 0:
        print(f"  ⚠️  Encontrados {nan_y} valores NaN no rótulo! Removendo linhas...")
        mask = y.notnull()
        X = X[mask]
        y = y[mask]

    num_atributos = X.shape[1]
    print(f"  📊 {X.shape[0]} amostras | {num_atributos} atributos")

    f1_kfold = {}
    f1_holdout = {}

    for k in k_values:
        # MÉTODO 1: 10-Fold Cross-Validation
        knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
        y_pred_cv = cross_val_predict(knn, X, y, cv=kf)
        f1_cv = f1_score(y, y_pred_cv, average='weighted')
        f1_kfold[k] = round(f1_cv, 3)

        # MÉTODO 2: Holdout 70/30
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=1
        )
        knn2 = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
        knn2.fit(X_train, y_train)
        y_pred_holdout = knn2.predict(X_test)
        f1_ho = f1_score(y_test, y_pred_holdout, average='weighted')
        f1_holdout[k] = round(f1_ho, 3)

        print(f"  k={k:2d} | 10-fold: {f1_kfold[k]:.3f} | 70/30: {f1_holdout[k]:.3f}")

    # Guardar 10-fold CV
    linha_cv = {
        'Base': f"{base_nome.replace('.csv','')} ({num_atributos} atrib.)",
        'Treino/Teste': '10-fold CV'
    }
    for k in k_values:
        linha_cv[f'{k}k'] = f1_kfold[k]
    linha_cv['Media_Geral'] = round(mean(list(f1_kfold.values())), 4)
    resultados.append(linha_cv)

    # Guardar Holdout
    linha_ho = {
        'Base': f"{base_nome.replace('.csv','')} ({num_atributos} atrib.)",
        'Treino/Teste': '70/30'
    }
    for k in k_values:
        linha_ho[f'{k}k'] = f1_holdout[k]
    linha_ho['Media_Geral'] = round(mean(list(f1_holdout.values())), 4)
    resultados.append(linha_ho)

# ============================================================
# 4. MONTAR DATAFRAME
# ============================================================
df_resultados = pd.DataFrame(resultados)

# ============================================================
# 5. MÉDIA E DESVIO PADRÃO
# ============================================================
colunas_k = [f'{k}k' for k in k_values]

media_row = {'Base': 'MÉDIA =>', 'Treino/Teste': ''}
desvio_row = {'Base': 'DESV. PAD =>', 'Treino/Teste': ''}

for col in colunas_k:
    media_row[col] = round(df_resultados[col].mean(), 4)
    desvio_row[col] = round(df_resultados[col].std(), 4)

media_row['Media_Geral'] = ''
desvio_row['Media_Geral'] = ''

df_resultados = pd.concat([
    df_resultados,
    pd.DataFrame([media_row]),
    pd.DataFrame([desvio_row])
], ignore_index=True)

# ============================================================
# 6. EXIBIR E SALVAR
# ============================================================
print("\n\n" + "="*80)
print("TABELA COMPLETA DE RESULTADOS")
print("="*80)
print(df_resultados.to_string(index=False))

df_resultados.to_csv('resultados_knn_completo.csv', index=False, encoding='utf-8')
print("\n✅ Tabela salva em 'resultados_knn_completo.csv'")

# ============================================================
# 7. RANKING E TOP 12
# ============================================================
df_bases = pd.DataFrame(resultados)
df_bases['Media_Geral'] = df_bases['Media_Geral'].astype(float)
ranking = df_bases.groupby('Base')['Media_Geral'].mean().sort_values(ascending=False)

print("\n\n" + "="*80)
print("RANKING DAS BASES (da melhor para a pior)")
print("="*80)
for i, (base, media) in enumerate(ranking.items(), 1):
    marcador = " ✅ TOP 12" if i <= 12 else ""
    print(f"  {i:2d}º  {base:55s}  Média F1: {media:.4f}{marcador}")

top12 = ranking.head(12)

print("\n\n" + "="*80)
print("AS 12 MELHORES BASES SELECIONADAS")
print("="*80)
for i, (base, media) in enumerate(top12.items(), 1):
    print(f"  {i:2d}. {base}  →  Média F1: {media:.4f}")

# ============================================================
# 8. SALVAR RANKING
# ============================================================
df_ranking = ranking.reset_index()
df_ranking.columns = ['Base', 'Media_F1']
df_ranking['Selecionada'] = ['SIM' if i < 12 else 'NÃO' for i in range(len(df_ranking))]
df_ranking.to_csv('ranking_bases_knn.csv', index=False, encoding='utf-8')
print("\n✅ Ranking salvo em 'ranking_bases_knn.csv'")


Processando: HOG_128_8x8.csv
  📊 656 amostras | 8100 atributos
  k= 1 | 10-fold: 0.599 | 70/30: 0.633
  k= 2 | 10-fold: 0.522 | 70/30: 0.546
  k= 3 | 10-fold: 0.617 | 70/30: 0.652
  k= 4 | 10-fold: 0.602 | 70/30: 0.568
  k= 5 | 10-fold: 0.614 | 70/30: 0.594
  k= 6 | 10-fold: 0.596 | 70/30: 0.603
  k= 7 | 10-fold: 0.607 | 70/30: 0.604
  k= 8 | 10-fold: 0.625 | 70/30: 0.609
  k= 9 | 10-fold: 0.620 | 70/30: 0.601
  k=10 | 10-fold: 0.606 | 70/30: 0.584

Processando: HOG_128_16x16.csv
  📊 656 amostras | 1764 atributos
  k= 1 | 10-fold: 0.627 | 70/30: 0.674
  k= 2 | 10-fold: 0.592 | 70/30: 0.567
  k= 3 | 10-fold: 0.627 | 70/30: 0.664
  k= 4 | 10-fold: 0.622 | 70/30: 0.599
  k= 5 | 10-fold: 0.630 | 70/30: 0.665
  k= 6 | 10-fold: 0.618 | 70/30: 0.619
  k= 7 | 10-fold: 0.628 | 70/30: 0.690
  k= 8 | 10-fold: 0.631 | 70/30: 0.690
  k= 9 | 10-fold: 0.591 | 70/30: 0.699
  k=10 | 10-fold: 0.626 | 70/30: 0.671

Processando: HOG_128_32x32.csv
  📊 656 amostras | 324 atributos
  k= 1 | 10-fold: 0.651 |